# Task 6 : Using segmentation priors to improve model fitting

Segmentation-guided MAP priors for voxel-wise MWF estimation and spatial smoothing.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
from utils import load_preterm_cohort, mean_decay_curve, hard_seg
from models import bi_exp_fixed, fit_bi_fixed
from analysis import (compute_cohort_priors, voxelwise_mwf_maps,
                      smooth_within_tissue)
from plotting import plot_mwf_maps, plot_mwf_histograms

## 6.1 : Cohort and prior specification

In [ ]:
preterm, _ = load_preterm_cohort('../data', exclude_file='../data/preterm_exclusions.csv')
mu_cohort, sigma_cohort = compute_cohort_priors(preterm)
print("Data-derived cohort MWF priors per tissue:")
print(pd.concat([mu_cohort.rename('mu'), sigma_cohort.rename('sigma')], axis=1).round(3))

## 6.2 : Voxel-wise MWF maps: no prior vs warm-start vs MAP

In [ ]:
ds = preterm[0]
sl = ds['reg'].shape[2] // 2

# Estimate noise sigma from WM tissue-mean residual
wm = mean_decay_curve(ds, 3)
S0_wm, v_wm = fit_bi_fixed(ds['TE'], wm, 20.0, 80.0, v_init=0.10)
sigma_noise = float(np.std(wm - bi_exp_fixed(ds['TE'], S0_wm, v_wm, 20.0, 80.0), ddof=1))

t0 = time.time()
mwf_np, mwf_warm, mwf_map = voxelwise_mwf_maps(
    ds, sl, mu_cohort, sigma_cohort, sigma_noise)
print(f"3 MWF maps computed in {time.time()-t0:.1f} s")

plot_mwf_maps(
    [mwf_np, mwf_warm, mwf_map],
    ['no prior', 'warm-start', 'MAP prior'],
    ds, sl)

wm_mask = (hard_seg(ds)[:, :, sl] == 3) & ds['mask'][:, :, sl]
plot_mwf_histograms(
    [mwf_np, mwf_warm, mwf_map],
    ['no prior', 'warm-start', 'MAP prior'],
    wm_mask, mu_cohort['WM'], ds, sl)

## 6.3 : Spatial smoothing as secondary prior

In [ ]:
mwf_smooth = smooth_within_tissue(mwf_map, wm_mask, fwhm_voxels=2.0)
plot_mwf_maps([mwf_map, mwf_smooth], ['MAP (voxel)', 'MAP + smoothing'], ds, sl)

vals_raw = mwf_map[wm_mask]; vals_raw = vals_raw[np.isfinite(vals_raw)]
vals_sm = mwf_smooth[wm_mask]; vals_sm = vals_sm[np.isfinite(vals_sm)]
print(f"WM voxel SD: before = {np.std(vals_raw):.3f}, after = {np.std(vals_sm):.3f}")